OFFLINE LEGAL RAG OLLAMA

In [11]:
print(" zstd installed")

 zstd installed


In [12]:
!apt-get install -y -q zstd

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (7,193 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [13]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [14]:
!pip install -q \
    ollama \
    faiss-cpu \
    sentence-transformers \
    datasets \
    langchain \
    langchain-community \
    langchain-ollama

print("✅ Python libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
✅ Python libraries installed


Step-3

In [15]:
import subprocess, time, requests
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout = subprocess.DEVNULL,
    stderr = subprocess.DEVNULL
)
print(" Waiting for Ollama server....")
for i in range(30):
  try:
    if requests.get("http://localhost:11434").status_code == 200:
      print("Ollama server is running")
      break
  except Exception:
      time.sleep(2)
else:
      print("X server did not start-re-run this cell once")

 Waiting for Ollama server....
Ollama server is running


In [16]:
print(" Pulling mistral (~4 GB)")
!ollama pull mistral
print(" Mistral ready!")



 Pulling mistral (~4 GB)

 Mistral ready!


In [17]:
import ollama
resp = ollama.chat(
    model="mistral",
    messages = [{"role": "user", "content": "Hi"}]
)
print(" ", resp["message"]["content"])

   Hello! How can I assist you today?

Is there something specific you need help with or would like to know more about? Whether it's a question related to technology, science, art, or anything else, feel free to ask and I'll do my best to provide an informative and engaging response. 😊


In [18]:
conversation_history = []
print(" Chat with Mistral (type 'quit' to exit)\n")
while True:
  user_input = input("You: ").strip()
  if user_input.lower() in ["quit", "exit", "q"]:
    print(" Chat ended.")
    break
  if not user_input:
    continue
  conversation_history.append({"role"  : "user", "content" : user_input})
  response = ollama.chat(model = "mistral", messages=conversation_history)
  reply = response["message"]["content"]
  conversation_history.append({"role" : "assistant", "content" : reply})

  print(f"\n Mistral: {reply}\n")

 Chat with Mistral (type 'quit' to exit)

You: q
 Chat ended.


In [19]:
import pandas as pd
from datasets import load_dataset

print("Loading ECHR legal case law from HuggingFace...")
DATASET_LOADED = False

try:
    ds = load_dataset("coastalcph/lex_glue", "ecthr_a", split="train", trust_remote_code=True)
    df = ds.to_pandas()[["text"]].dropna()
    df["text"] = df["text"].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))
    df = df[df["text"].str.len() > 300].head(150).reset_index(drop=True)
    df["doc_id"] = [f"ECHR_Case_{i}" for i in range(len(df))]
    print(f"✅ Loaded {len(df)} real ECHR court cases")
    DATASET_LOADED = True
except Exception as e:
    print(f"⚠️  HuggingFace dataset unavailable ({e}) — using built-in corpus")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'coastalcph/lex_glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'coastalcph/lex_glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading ECHR legal case law from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ecthr_a/train-00000-of-00001.parquet:   0%|          | 0.00/42.4M [00:00<?, ?B/s]

...

ecthr_a/validation-00000-of-00001.parque(…):   0%|          | 0.00/5.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Loaded 150 real ECHR court cases


In [20]:
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-ollama langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 9.5 MB/s eta 0:00:00


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter= RecursiveCharacterTextSplitter(chunk_size=450,chunk_overlap=80)
documents=[]


for _, row in df.iterrows():
    for i, chunk in enumerate(splitter.split_text(row["text"])):
        documents.append(
            Document(
                page_content=chunk,
                metadata={"source": row["doc_id"], "chunk": i}
            )
        )

print(f"📄 {len(documents)} chunks from {len(df)} documents")

print("Embedding on GPU...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore.save_local("content/legal_faiss")

print(f"✅ FAISS index ready - {vectorstore.index.ntotal} vectors")

📄 5335 chunks from 150 documents
Embedding on GPU...


/tmp/ipykernel_1907/3607965163.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS index ready - 5335 vectors


In [22]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="mistral", temperature=0.1, num_predict=400)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

prompt = PromptTemplate.from_template("""
You are an expert legal research assistant.
Answer the question using ONLY the legal documents provided below.
Cite the document names in your answer when referencing specific cases or principles.
If the documents do not contain enough information, say so clearly.

Legal Documents:
{context}

Question: {question}

Legal Answer:""")

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print("✅ RAG chain ready — Mistral + FAISS + Local Embeddings")

✅ RAG chain ready — Mistral + FAISS + Local Embeddings


In [23]:
def ask(question: str):
    print(f"\n{'='*65}")
    print(f"❓ {question}")
    print(f"{'='*65}")
    docs = retriever.invoke(question)           # ← was get_relevant_documents
    sources = list(dict.fromkeys(d.metadata["source"] for d in docs))
    print(f"📚 Sources pulled: {', '.join(sources)}")
    print("-"*65)
    answer = rag_chain.invoke(question)
    print(f"⚖️  {answer}")
    return answer


_ = ask("What rights must police inform a suspect of before interrogation?")



❓ What rights must police inform a suspect of before interrogation?
📚 Sources pulled: ECHR_Case_30, ECHR_Case_119
-----------------------------------------------------------------
⚖️   The provided documents do not contain specific information regarding what rights police must inform a suspect of before interrogation. However, they do discuss the concept of reasonable suspicion and the burden of proof for demonstrating the legality of an arrest. It can be inferred that the court considers it necessary for the police to have reasonable grounds or sufficient information to suspect a person before exercising the power of arrest ([ECHR_Case_30]). However, the documents do not provide details about what rights must be informed to a suspect during interrogation. For a comprehensive answer regarding the rights that must be informed to a suspect during interrogation, additional legal documents or case law would be required.


In [24]:
!pip install -q gradio

In [25]:
import gradio as gr


# ── helpers ──────────────────────────────────────────────────
def get_sources(docs):
    return list(dict.fromkeys(d.metadata["source"] for d in docs))


def rag_response(message, history):
    docs   = retriever.invoke(message)
    sources = get_sources(docs)
    context = "\n\n".join(
        f"[{d.metadata['source']}]\n{d.page_content}" for d in docs
    )


    prompt_text = f"""You are an expert legal research assistant.
Answer the question using ONLY the legal documents provided below.
Cite the document names in your answer when referencing specific cases or principles.
If the documents do not contain enough information, say so clearly.


Legal Documents:
{context}


Question: {message}


Legal Answer:"""


    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt_text}]
    )
    answer = response["message"]["content"]


    source_line = "\n\n---\n📚 *Sources:* " + " · ".join(f"{s}" for s in sources)
    return answer + source_line


# ── UI ───────────────────────────────────────────────────────
with gr.Blocks(title="Legal RAG", theme=gr.themes.Soft()) as demo:


    gr.Markdown("# Legal RAG — Powered by Mistral + FAISS\nAsk any legal question. Answers are grounded in the case law documents.")


    chatbot = gr.Chatbot(
        label="Legal Assistant",
        height=480,
        show_copy_button=True,
        render_markdown=True,
    )
    with gr.Row():
        txt = gr.Textbox(
            placeholder="e.g. What are Miranda rights?",
            show_label=False,
            scale=9,
        )
        btn = gr.Button("Ask", scale=1, variant="primary")


    gr.Examples(
        examples=[
            "What rights must police inform a suspect of before interrogation?",
            "What are the elements required to prove negligence?",
            "What is the difference between Chapter 7 and Chapter 13 bankruptcy?",
            "What fiduciary duties do corporate directors owe to shareholders?",
            "How does the Fourth Amendment protect against unreasonable searches?",
            "Can corporations spend unlimited money on political campaigns?",
        ],
        inputs=txt,
    )


    def respond(message, chat_history):
        answer = rag_response(message, chat_history)
        chat_history.append((message, answer))
        return "", chat_history


    txt.submit(respond, [txt, chatbot], [txt, chatbot])
    btn.click(respond,  [txt, chatbot], [txt, chatbot])


demo.launch(share=True, quiet=True)

/tmp/ipykernel_1907/1895815480.py:45: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Legal RAG", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1907/1895815480.py:51: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1907/1895815480.py:51: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1907/1895815480.py:51: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to e

* Running on public URL: https://92a2f5a78c5025e5a4.gradio.live
